### Mittag-Leffler's Theorem: Rebuilding a Function From Its Poles

Everything in this notebook is done to one concrete function:

$$
f(z) = 1.5
+ \frac{1.0 + 0.5i}{z - (1.0 + 1.0i)}
+ \frac{-0.8 + 0.3i}{z - (-1.5 + 0.5i)}
+ \frac{0.6 - 0.9i}{z - (0.5 - 1.5i)}
$$

It is written in exactly this form so that nothing about it is a mystery: three
simple poles at $1.0 + 1.0i$, $-1.5 + 0.5i$ and $0.5 - 1.5i$, one number sitting
on top of each, and a constant $1.5$ that the three fractions are added to. This
is the same expression that appears as `f_exact()` in the first code cell.

The goal is to throw nearly all of that away and get it back. Mittag-Leffler's
theorem says the entire function can be rebuilt from nothing but the locations of
its poles, the three numbers on top of them, and the single value $f(0)$. Watch
for what is *not* on that list: the constant $1.5$ is never touched again once
`f_exact()` is defined, except at the very end, where it is printed purely as an
answer key for the value the reconstruction recovers on its own. And $1.5$ is not
$f(0)$ either, since all three fractions contribute at the origin as well - the
first code cell works out that $f(0) = -0.45 + 0.09i$.

A function is **meromorphic** on a region if it is analytic everywhere except at
isolated **poles**, the points where it blows up. Near a simple pole $z_j$ the
function behaves like

$$
f(z) \approx \frac{b_j}{z - z_j},
$$

and that piece is called the **principal part** at $z_j$. The number $b_j$ is the
**residue**: it says how strong the pole is and in which complex direction it
pushes. For the function above the residues are read straight off the numerators,
$b_1 = 1.0 + 0.5i$, $b_2 = -0.8 + 0.3i$ and $b_3 = 0.6 - 0.9i$, because it was
built by adding up its own principal parts in the first place.

Mittag-Leffler's theorem says something remarkable. A meromorphic function is
nothing more than the sum of its principal parts plus an analytic leftover. For a
function with finitely many simple poles $z_1, \ldots, z_M$ (none at the origin)
whose growth is mild enough that $f(z)/z \to 0$ as $|z| \to \infty$, the theorem
takes the concrete form shown below:

$$
f(z) = f(0) + \sum_{j=1}^{M} b_j \left( \frac{1}{z - z_j} + \frac{1}{z_j} \right)
$$

Read the right-hand side as a recipe. Start with the single number $f(0)$, then
add one correction per pole. Each correction is the pole's principal part
$b_j/(z - z_j)$, shifted by the constant $b_j/z_j$ so that the whole correction
vanishes at $z = 0$ and does not disturb the $f(0)$ we started from.

Rather than checking the finished formula all at once, this notebook builds the
function up **one pole at a time**:

1. Look at each principal part on its own.
2. Add the poles in stages and watch the curve snap onto the true function.
3. Plot the error left over after each stage, on the real axis and then across
   the complex plane.
4. Confirm that once every principal part is removed, what remains is analytic.

In [ ]:
"""mittag_leffler.ipynb"""

# Cell 01 - The meromorphic function, its poles and its residues

%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np

# The test function is built by hand so we know its poles and residues exactly:
#     f(z) = ANALYTIC_PART + sum_j RESIDUES[j] / (z - POLES[j])
# Nothing below ever peeks at ANALYTIC_PART - the reconstruction only gets to
# use the poles, the residues, and the single number f(0).
ANALYTIC_PART = 1.5

POLES = np.array([1.0 + 1.0j, -1.5 + 0.5j, 0.5 - 1.5j], dtype=complex)
RESIDUES = np.array([1.0 + 0.5j, -0.8 + 0.3j, 0.6 - 0.9j], dtype=complex)


def f_exact(z: complex | np.ndarray) -> complex | np.ndarray:
    """Return the exact meromorphic function with three simple poles."""
    total = np.full_like(np.asarray(z, dtype=complex), ANALYTIC_PART, dtype=complex)
    for pole, residue in zip(POLES, RESIDUES):
        total = total + residue / (z - pole)
    return total


def format_complex(value: complex, places: int = 3) -> str:
    """Return a complex number as a compact 'a + bi' string."""
    # Adding 0.0 after rounding turns a -0.0 back into +0.0, so a value that is
    # zero to the printed precision never displays as "-0.000"
    real = round(value.real, places) + 0.0
    imag = round(value.imag, places) + 0.0
    sign = "+" if imag >= 0 else "-"
    return f"{real:+.{places}f} {sign} {abs(imag):.{places}f}i"


# Print the pole/residue table the reconstruction is allowed to see
f_at_zero = complex(f_exact(0.0 + 0.0j))

print("Pole z_j              Residue b_j")
print("-" * 44)
for pole, residue in zip(POLES, RESIDUES):
    print(f"{format_complex(pole):>20}  {format_complex(residue):>20}")
print("-" * 44)
print(f"f(0) = {format_complex(f_at_zero)}")

---
### The building blocks: one principal part per pole

Each pole contributes exactly one term to the sum,

$$
p_j(z) = b_j \left( \frac{1}{z - z_j} + \frac{1}{z_j} \right),
$$

and the partial reconstruction after $n$ stages is just $f(0)$ plus the first
$n$ of them:

$$
f_n(z) = f(0) + \sum_{j=1}^{n} p_j(z)
$$

so $f_0(z) = f(0)$ is a flat constant and $f_3(z)$ should be the function itself.

The constant $1/z_j$ inside $p_j$ is the part students usually skip over, so the
cell below tests it directly. Substituting $z = 0$ gives
$b_j(1/(0 - z_j) + 1/z_j) = 0$, meaning **every principal part is exactly zero at
the origin**. That is what lets the terms be added in any order without ever
spoiling the value $f(0)$ that anchors the whole formula.

In [ ]:
# Cell 02 - Principal parts and partial reconstructions


def principal_part(
    z: complex | np.ndarray, pole: complex, residue: complex
) -> complex | np.ndarray:
    """Return the shifted principal part b*(1/(z - z_j) + 1/z_j) of one pole."""
    return residue * (1.0 / (z - pole) + 1.0 / pole)


def partial_reconstruction(
    z: complex | np.ndarray, n_terms: int
) -> complex | np.ndarray:
    """Return f_n(z) = f(0) plus the principal parts of the first n poles."""
    total = np.full_like(np.asarray(z, dtype=complex), f_at_zero, dtype=complex)
    for pole, residue in zip(POLES[:n_terms], RESIDUES[:n_terms]):
        total = total + principal_part(z, pole, residue)
    return total


# Check 1: each principal part vanishes at the origin, by construction
for index, (pole, residue) in enumerate(zip(POLES, RESIDUES), start=1):
    value = complex(principal_part(0.0 + 0.0j, pole, residue))
    print(f"p_{index}(0) = {format_complex(value)}   (expected 0.000 + 0.000i)")

# Check 2: with every pole included, f_3 should reproduce f at a test point
z_test = 2.0 - 0.75j
print()
print(
    f"f(z)   at z = {format_complex(z_test)}: {format_complex(complex(f_exact(z_test)))}"
)
print(
    f"f_3(z) at z = {format_complex(z_test)}: "
    f"{format_complex(complex(partial_reconstruction(z_test, 3)))}"
)

---
### Step 1: look at each principal part on its own

Before adding anything, here is what the three terms $p_1$, $p_2$ and $p_3$ look
like along the real axis. All three poles sit off the real axis, so none of these
curves actually blows up here - each one is a smooth bump whose height and width
are set by the residue $b_j$ and by how close the pole $z_j$ comes to the real
line.

Notice that all three curves pass through zero at $x = 0$, exactly as the check
in the previous cell predicted, and that each one flattens out to the constant
$b_j / z_j$ far from the origin.

In [ ]:
# Cell 03 - The three principal parts plotted separately along the real axis

# An odd point count puts a sample exactly on x = 0, where every principal
# part vanishes and the reconstruction must agree with f(0) to the last bit
x = np.linspace(-4.0, 4.0, 801)
z_line = x.astype(complex)

fig, (ax_real, ax_imag) = plt.subplots(1, 2, figsize=(12, 4.5))

for index, (pole, residue) in enumerate(zip(POLES, RESIDUES), start=1):
    term = principal_part(z_line, pole, residue)
    label = f"$p_{index}$, pole at {pole:.1f}"
    ax_real.plot(x, term.real, lw=2, label=label)
    ax_imag.plot(x, term.imag, lw=2, label=label)

for axis, part in ((ax_real, "Real"), (ax_imag, "Imaginary")):
    axis.axhline(0.0, color="gray", lw=1)
    axis.axvline(0.0, color="gray", lw=1)
    axis.set_title(f"{part} part of each principal part")
    axis.set_xlabel("x, where z = x")
    axis.set_ylabel(f"{part} part")
    axis.grid(True, alpha=0.3)
    axis.legend(framealpha=1.0, facecolor="white", fontsize=9)

fig.tight_layout()
plt.show()

---
### Step 2: add the poles one at a time

Now the terms go in one after another. Stage 0 is the flat line $f(0)$, with no
pole information at all. Stage 1 adds $p_1$, stage 2 adds $p_2$, and stage 3 adds
$p_3$.

Each panel draws the exact function as a thick gray curve and the partial
reconstruction $f_n$ on top of it. The dotted curve is the term that was just
switched on, and it is exactly the change that term makes, since
$f_n - f_{n-1} = p_n$. Notice that the first two additions leave the blue curve
still visibly off the gray one: a partial sum of principal parts is *not* a good
approximation, it is simply an incomplete one. Only when the last pole goes in
does the reconstruction land on the exact curve, and then it lands to the last
bit of double precision.

In [ ]:
# Cell 04 - Staged buildup of the reconstruction along the real axis

exact_line = f_exact(z_line)

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)

for n_terms, axis in enumerate(axes.flat):
    approx = partial_reconstruction(z_line, n_terms)

    axis.plot(x, exact_line.real, lw=6, color="lightgray", label="Exact Re[f(z)]")
    axis.plot(x, approx.real, lw=2, color="tab:blue", label=f"Re[$f_{n_terms}$(z)]")

    # The term that was just switched on, drawn on its own for comparison
    if n_terms > 0:
        newest = principal_part(z_line, POLES[n_terms - 1], RESIDUES[n_terms - 1])
        axis.plot(
            x,
            newest.real,
            ls=":",
            lw=2,
            color="tab:red",
            label=f"term just added: $p_{n_terms}$",
        )

    max_error = np.max(np.abs(exact_line - approx))
    axis.set_title(f"Stage {n_terms}: {n_terms} pole(s), max error = {max_error:.2e}")
    axis.grid(True, alpha=0.3)
    axis.legend(loc="upper right", framealpha=1.0, facecolor="white", fontsize=9)

for axis in axes[1]:
    axis.set_xlabel("x, where z = x")
for axis in axes[:, 0]:
    axis.set_ylabel("Real part")

# Extra headroom at the top so the legends do not sit on the curves
low, high = exact_line.real.min(), exact_line.real.max()
axes[0, 0].set_ylim(low - 0.4, high + 1.2)

fig.suptitle("Mittag-Leffler Reconstruction, One Pole at a Time")
fig.tight_layout()
plt.show()

---
### Step 3: the error left after each stage

Eyeballing the panels above only goes so far, so the honest way to track progress
is to plot the leftover error

$$
E_n(x) = \bigl| f(x) - f_n(x) \bigr|
$$

on a logarithmic axis. The result is not the gentle staircase you might expect.
The first two poles buy very little: the error stays of order one, shrinking only
from about 2.1 to 0.7. Then the third pole drops it fifteen orders of magnitude
in a single step, down to roughly $10^{-16}$.

That cliff is the whole character of the finite Mittag-Leffler formula. It is not
a series that converges gradually toward $f$; it is an **identity** that is either
incomplete or exact. Leave out one principal part and its pole is still missing
from the reconstruction, no matter how far away you sample. Put every one in and
the remaining difference is not approximation error at all - it is nothing but the
floating-point rounding of the arithmetic, which is why the final curve sits right
at machine epsilon.

Every curve also plunges at $x = 0$. That spike is the $1/z_j$ shift doing its job:
all the principal parts vanish at the origin, so even stage 0 reproduces $f(0)$ to
machine precision. The sample grid deliberately includes $x = 0$ exactly, so the
spike reaches all the way down.

In [ ]:
# Cell 05 - Remaining error on the real axis after each stage

fig, ax = plt.subplots(figsize=(10, 5.5))

for n_terms in range(len(POLES) + 1):
    error = np.abs(exact_line - partial_reconstruction(z_line, n_terms))
    # Floor the error so log scaling survives a difference of exactly zero
    ax.semilogy(
        x,
        np.maximum(error, 1.0e-17),
        lw=2,
        label=f"after {n_terms} pole(s), max = {np.max(error):.3e}",
    )

ax.axhline(np.finfo(float).eps, color="gray", ls="--", lw=1.5, label="machine epsilon")

ax.set_title("Error Remaining After Each Stage of the Expansion")
ax.set_xlabel("x, where z = x")
ax.set_ylabel(r"$|f(x) - f_n(x)|$")
ax.set_ylim(1.0e-17, 1.0e2)
ax.grid(True, which="both", alpha=0.3)
ax.legend(framealpha=1.0, facecolor="white")

fig.tight_layout()
plt.show()

print(
    f"Maximum error with all {len(POLES)} poles included: "
    f"{np.max(np.abs(exact_line - partial_reconstruction(z_line, len(POLES)))):.3e}"
)

---
### Step 4: where the poles are and how strong they are

The real axis is only a one-dimensional slice through a two-dimensional story.
This map shows the poles in the complex plane, with the sample line of the plots
above drawn across the middle.

Each residue $b_j$ is itself a complex number, so it carries a magnitude and a
direction. The marker area is scaled by $|b_j|$ and the arrow points along
$\arg b_j$. That direction is not decorative: it sets the phase of the swirl that
the pole imprints on the function around it.

In [ ]:
# Cell 06 - Map of the poles and their residues in the complex plane

# Label positions relative to each pole, hand-placed so no label sits on an arrow
LABEL_OFFSETS = [(-0.85, 0.90), (-1.40, -1.05), (-1.55, -0.05)]

fig, ax = plt.subplots(figsize=(7, 7))

ax.axhline(0.0, color="gray", lw=1)
ax.axvline(0.0, color="gray", lw=1)
ax.plot([-4, 4], [0, 0], color="tab:green", lw=2, alpha=0.5, label="real-axis sample")
ax.plot(0.0, 0.0, "ks", ms=8, label="origin, where $f(0)$ is read off")

for index, (pole, residue) in enumerate(zip(POLES, RESIDUES), start=1):
    offset_x, offset_y = LABEL_OFFSETS[index - 1]
    ax.scatter(
        pole.real,
        pole.imag,
        s=600 * abs(residue),
        color=f"C{index - 1}",
        alpha=0.5,
        edgecolors="black",
        zorder=3,
    )
    ax.annotate(
        f"$z_{index}$ = {pole:.1f}\n$b_{index}$ = {residue:.1f}",
        xy=(pole.real, pole.imag),
        xytext=(pole.real + offset_x, pole.imag + offset_y),
        fontsize=9,
    )
    # Arrow along arg(b_j), scaled by |b_j|, showing the residue as a vector
    ax.arrow(
        pole.real,
        pole.imag,
        residue.real,
        residue.imag,
        width=0.02,
        head_width=0.12,
        length_includes_head=True,
        color=f"C{index - 1}",
        zorder=4,
    )

ax.set_title("Poles (area $\\propto |b_j|$) and Residues (arrows)")
ax.set_xlabel("Re(z)")
ax.set_ylabel("Im(z)")
ax.set_xlim(-3.0, 3.0)
ax.set_ylim(-3.0, 3.0)
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left", framealpha=1.0, facecolor="white", fontsize=9)

fig.tight_layout()
plt.show()

---
### Step 5: watching the poles disappear across the complex plane

The real axis is only one slice, and the semilog plot showed that the reconstruction
error stays stubbornly of order one until the very last term. So plotting that same
error across the plane would just show three flat panels and then a dark one.

The quantity that really does change one stage at a time is the **remainder** left
after peeling off the principal parts collected so far:

$$
g_n(z) = f(z) - \sum_{j=1}^{n} \frac{b_j}{z - z_j}
$$

$g_n$ still has poles at the $M - n$ places not yet removed, and it is perfectly
smooth at the ones already handled. Plotting $\log_{10}|g_n(z)|$ makes that
literal: each bright singular spike winks out as its principal part is subtracted,
and the poles already dealt with leave no scar behind at all. By stage 3 the whole
plane is a single flat color, which is the picture of a function with no poles
left anywhere.

In [ ]:
# Cell 07 - Stripping the principal parts off, one pole at a time


def mask_near_poles(
    values: np.ndarray, z_grid: np.ndarray, radius: float = 0.12
) -> np.ma.MaskedArray:
    """Mask grid points inside small disks around the poles."""
    mask = np.zeros(z_grid.shape, dtype=bool)
    for pole in POLES:
        mask |= np.abs(z_grid - pole) < radius
    return np.ma.array(values, mask=mask)


def stripped_remainder(z: np.ndarray, n_terms: int) -> np.ndarray:
    """Return f(z) with the raw principal parts of the first n poles removed."""
    total = np.asarray(f_exact(z), dtype=complex).copy()
    for pole, residue in zip(POLES[:n_terms], RESIDUES[:n_terms]):
        total = total - residue / (z - pole)
    return total


# An odd point count keeps every sample off the poles themselves
axis_values = np.linspace(-3.0, 3.0, 401)
grid_x, grid_y = np.meshgrid(axis_values, axis_values)
z_grid = grid_x + 1j * grid_y

exact_grid = f_exact(z_grid)

fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))

# A sample landing exactly on a pole would divide by zero; the color scale
# clips such a point to the top of the range, which is where it belongs
with np.errstate(divide="ignore", invalid="ignore"):
    for n_terms, axis in enumerate(axes):
        magnitude = np.log10(np.abs(stripped_remainder(z_grid, n_terms)))

        image = axis.imshow(
            magnitude,
            extent=[-3.0, 3.0, -3.0, 3.0],
            origin="lower",
            aspect="equal",
            cmap="magma",
            vmin=-1.0,
            vmax=1.5,
        )

        # Poles already stripped are white dots; those still present are red
        axis.scatter(
            POLES[:n_terms].real,
            POLES[:n_terms].imag,
            marker="o",
            facecolors="none",
            edgecolors="white",
            s=90,
            linewidths=1.5,
        )
        axis.scatter(
            POLES[n_terms:].real,
            POLES[n_terms:].imag,
            marker="x",
            c="cyan",
            s=80,
            linewidths=2,
        )

        axis.set_title(f"Stage {n_terms}: $g_{n_terms}(z)$")
        axis.set_xlabel("Re(z)")

axes[0].set_ylabel("Im(z)")

colorbar = fig.colorbar(image, ax=axes, fraction=0.02, pad=0.02)
colorbar.set_label(r"$\log_{10}|g_n(z)|$")

fig.suptitle(
    "Principal Parts Removed One at a Time "
    "(cyan x = pole still present, white circle = already stripped)"
)
plt.show()

---
### Step 6: the leftover is analytic, and the finished formula is exact

The flat final panel deserves a number attached to it. The cell below measures how
far the fully stripped remainder $g_3$ varies across the grid, and the answer is
that it does not vary at all beyond rounding: it is the constant
`ANALYTIC_PART` that the poles were bolted onto when the test function was built,
recovered without the reconstruction ever being told what it was.

That is the payoff of the theorem. Once the principal parts are removed, what is
left is **entire** - analytic on the whole plane, no poles anywhere. And when the
leftover is this tame, one sample of it is enough to identify it, which is exactly
the role $f(0)$ plays in the formula at the top of the notebook.

The second plot closes the loop by mapping the error of the finished
reconstruction, $\log_{10}|f(z) - f_3(z)|$, over the same patch. It sits on the
$10^{-16}$ floor nearly everywhere, so the identity holds across the plane and not
just on the real axis sampled earlier. Small disks around the poles are masked
out, since both sides diverge there and the difference of two infinities is not
meaningful, and you can see a faint halo just outside each mask: close to a pole
both sides are enormous, and rounding error scales with the size of the numbers
being subtracted, so the *absolute* error creeps up to a few times $10^{-15}$
there even though the *relative* error never does.

In [ ]:
# Cell 08 - The analytic remainder, and the error of the finished formula

# Ignore the masked disks, where the subtraction is a difference of two infinities
remainder_masked = mask_near_poles(stripped_remainder(z_grid, len(POLES)), z_grid)
remainder_mean = complex(np.mean(remainder_masked))
spread = np.max(np.abs(remainder_masked - remainder_mean))

print(f"Remainder g_3(z), averaged over the grid : {format_complex(remainder_mean)}")
print(f"Largest deviation from that average      : {spread:.3e}")
print(f"Constant used to build f(z)              : {ANALYTIC_PART:+.3f}")
print()

grid_error = mask_near_poles(
    np.abs(exact_grid - partial_reconstruction(z_grid, len(POLES))), z_grid
)
print(f"Largest |f(z) - f_3(z)| over the grid    : {np.max(grid_error):.3e}")
print(f"Machine epsilon for float64              : {np.finfo(float).eps:.3e}")

fig, ax = plt.subplots(figsize=(7.5, 6))

# Floor at 1e-16 so the log scale bottoms out at machine precision
image = ax.imshow(
    np.ma.log10(np.maximum(grid_error, 1.0e-16)),
    extent=[-3.0, 3.0, -3.0, 3.0],
    origin="lower",
    aspect="equal",
    cmap="viridis",
    vmin=-16.0,
    vmax=0.0,
)
ax.scatter(
    POLES.real, POLES.imag, marker="x", c="red", s=80, linewidths=2, label="poles"
)

ax.set_title("Error of the Completed Reconstruction Across the Plane")
ax.set_xlabel("Re(z)")
ax.set_ylabel("Im(z)")
ax.legend(loc="upper left", framealpha=1.0, facecolor="white")

colorbar = fig.colorbar(image, ax=ax)
colorbar.set_label(r"$\log_{10}|f(z) - f_3(z)|$")

fig.tight_layout()
plt.show()

---
### Putting the stages in motion

The same four stages, played as an animation. The left panel shows the
reconstruction climbing onto the exact curve; the right panel shows the error
hovering near one for two stages and then falling off a cliff to machine
precision the moment the last pole goes in.

The animation is rendered as JavaScript frames with `to_jshtml()`, which works
under the `%matplotlib inline` backend used by the rest of this notebook. Use the
play button, or step through frame by frame to read each stage.

In [ ]:
# Cell 09 - Animate the stages with a JavaScript player

from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation

fig, (ax_curve, ax_error) = plt.subplots(1, 2, figsize=(12, 4.5))

ax_curve.plot(x, exact_line.real, lw=6, color="lightgray", label="Exact Re[f(z)]")
(curve_line,) = ax_curve.plot([], [], lw=2, color="tab:blue", label="Re[$f_n$(z)]")
ax_curve.set_xlim(x.min(), x.max())
ax_curve.set_ylim(exact_line.real.min() - 0.5, exact_line.real.max() + 0.5)
ax_curve.set_xlabel("x, where z = x")
ax_curve.set_ylabel("Real part")
ax_curve.grid(True, alpha=0.3)
ax_curve.legend(loc="upper right", framealpha=1.0, facecolor="white")

(error_line,) = ax_error.semilogy([], [], lw=2, color="tab:red")
ax_error.set_xlim(x.min(), x.max())
ax_error.set_ylim(1.0e-17, 1.0e2)
ax_error.set_xlabel("x, where z = x")
ax_error.set_ylabel(r"$|f(x) - f_n(x)|$")
ax_error.grid(True, which="both", alpha=0.3)


def draw_stage(n_terms: int) -> tuple:
    """Draw one animation frame showing the reconstruction after n poles."""
    approx = partial_reconstruction(z_line, n_terms)
    error = np.abs(exact_line - approx)

    curve_line.set_data(x, approx.real)
    error_line.set_data(x, np.maximum(error, 1.0e-17))

    ax_curve.set_title(f"Stage {n_terms}: {n_terms} of {len(POLES)} poles added")
    ax_error.set_title(f"Remaining error, max = {np.max(error):.2e}")
    return curve_line, error_line


animation = FuncAnimation(
    fig, draw_stage, frames=len(POLES) + 1, interval=1600, repeat=True
)

# Close the static figure so only the animated player is displayed
plt.close(fig)
display(HTML(animation.to_jshtml()))